# Trigger `iceberg_migration` DAG — WF-374 In-Place Text CTAS End-to-End Test

One-shot harness for **WF-374**: text tables (`LazySimpleSerDe` + `TextInputFormat`) migrated
**in place** to Iceberg via a full CTAS copy plus a name swap, so the table keeps its original
`db.table` identity and no consumer has to be repointed.

### What the feature does

With `inplace_migration = T` and `iceberg_inplace_text_ctas` on, an **EXTERNAL** text table is
migrated as:

```
CREATE TABLE {db}.{tbl}__ice_staging USING iceberg [PARTITIONED BY (...)]
  LOCATION '<original location>_iceberg'
  TBLPROPERTIES ('write.spark.fanout.enabled'='true')
  AS SELECT * FROM {db}.{tbl}
--- verify against the still-live original: row count, then normalized schema ---
ALTER TABLE {db}.{tbl}              RENAME TO {db}.{tbl}_backup_
ALTER TABLE {db}.{tbl}__ice_staging RENAME TO {db}.{tbl}
```

Tracking records `migration_type = 'INPLACE_CTAS'`. The original text table survives as
`{tbl}_backup_`; its files are **never** purged by the DAG.

### Routing matrix under test

| `source_format` | `inplace` | flag | `table_type` | expected |
|---|---|---|---|---|
| PARQUET / ORC / AVRO | T | any | any | `system.migrate`, zero-copy, `INPLACE` |
| TEXT | T | on | EXTERNAL | CTAS swap, `INPLACE_CTAS` |
| TEXT | T | on | not confirmed EXTERNAL | SKIPPED `MANAGED_TEXT_INPLACE_UNSUPPORTED` |
| TEXT | T | off | any | SKIPPED `TEXT_FORMAT_INPLACE_UNSUPPORTED` |
| TEXT | F | any | any | snapshot CTAS into `{db}_iceberg`, unchanged |

### Prerequisites

- A JupyterHub Spark session on the tenant, with Ranger grants to create/rename/drop in the
  test databases (Step 4 probes this and reports what is missing).
- `AIRFLOW_BASE_URL`, `AIRFLOW_USERNAME`, `AIRFLOW_PASSWORD` — the in-cluster URL bypasses
  Keycloak.
- `DAG_SUFFIX` matching whatever you passed to `deploy.py`, and `S3_BUCKET`.
- The WF-374 branch deployed: `python deploy.py --project migrator --dag iceberg --owner <you> --suffix <suffix>`

### Order of play

**Step 2 is the gate.** It probes six runtime assumptions the design rests on, none of which can
be verified from the repo because no Iceberg version is pinned there. If any fails, stop — the
rest of the notebook is meaningless and the design itself needs revisiting.

In [ ]:
# ── Step 0 — Configuration ──────────────────────────────────────────────────────
import os

# --- Airflow REST API (in-cluster URL bypasses Keycloak; FAB auth -> JWT via /auth/token) ---
AIRFLOW_BASE_URL = os.environ.get("AIRFLOW_BASE_URL", "")
AIRFLOW_USERNAME = os.environ.get("AIRFLOW_USERNAME", "")
AIRFLOW_PASSWORD = os.environ.get("AIRFLOW_PASSWORD", "")

# deploy.py always appends --suffix to the dag_id, so a deployed copy is
# `iceberg_migration_<suffix>`. Leave DAG_SUFFIX unset only if an unsuffixed DAG exists.
DAG_SUFFIX = os.environ.get("DAG_SUFFIX", "")
DAG_ID     = "iceberg_migration" + (f"_{DAG_SUFFIX}" if DAG_SUFFIX else "")

# --- S3 surface ---
BUCKET        = os.environ.get("S3_BUCKET", "")
TENANT_PREFIX = os.environ.get("S3_TENANT_PREFIX", f"s3a://{BUCKET}/es-tenant-2")

_tag       = DAG_SUFFIX or "e2e"
TEST_BASE  = f"{TENANT_PREFIX}/wf374_text_ctas_{_tag}"
EXCEL_DIR  = f"{TENANT_PREFIX}/configs"

# --- Databases ---
# In-place migration writes back into the SOURCE database, so TEST_DB is both source and
# destination. SNAP_DB exercises snapshot mode, which still lands in `{SNAP_DB}_iceberg`.
TEST_DB  = os.environ.get("TEST_DB",  f"wf374_inplace_{_tag}")
SNAP_DB  = os.environ.get("SNAP_DB",  f"wf374_snapshot_{_tag}")
PROBE_DB = os.environ.get("PROBE_DB", f"wf374_probe_{_tag}")

# --- Deployed bundle on S3 (DEPLOY_DAGS_PREFIX from env.shared; sits at the BUCKET ROOT) ---
DAGS_S3_PREFIX = os.environ.get("DEPLOY_DAGS_PREFIX", "airflow/es-tenant-2/dags")
CONFIG_S3_DIR  = f"s3a://{BUCKET}/{DAGS_S3_PREFIX.strip('/')}/migrator_utils/migration_configs"

# TRACKING_DB / REPORT_LOCATION are deliberately NOT set here — Step 3 binds them from the
# deployed env file, so the notebook cannot drift from what the DAG resolves.

print("Configuration loaded.")
print(f"  Airflow URL    : {AIRFLOW_BASE_URL or '(unset — set AIRFLOW_BASE_URL)'}")
print(f"  DAG ID         : {DAG_ID}" + ("" if DAG_SUFFIX else "  (no DAG_SUFFIX set)"))
print(f"  Test base      : {TEST_BASE}")
print(f"  In-place DB    : {TEST_DB}   (source == destination)")
print(f"  Snapshot DB    : {SNAP_DB}   (destination is {SNAP_DB}_iceberg)")
print(f"  Probe DB       : {PROBE_DB}")
print(f"  Deployed config: {CONFIG_S3_DIR}")

## Step 1 — Helpers

Spark session, Hadoop-FS helpers, Airflow JWT REST client, and a handful of metastore
accessors (`location_of`, `provider_of`, `table_type_of`) that the assertions lean on.

In [ ]:
# ── Step 1 — Helpers ────────────────────────────────────────────────────────────
import requests
from py4j.java_gateway import java_import
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder \
        .appName("wf374-text-ctas-e2e") \
        .enableHiveSupport() \
        .getOrCreate()
    print(f"Created Spark session (version {spark.version})")

spark.sparkContext.setLogLevel("WARN")
java_import(spark._jvm, "org.apache.hadoop.fs.*")

# Text tables copied from MapR often keep nested sub-directories under a partition path;
# the DAG sets these too, so the notebook's own counts match what the DAG sees.
spark.conf.set("mapreduce.input.fileinputformat.input.dir.recursive", "true")
spark.conf.set("mapred.input.dir.recursive", "true")


def _fs(path):
    return spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path), spark._jsc.hadoopConfiguration()
    )


def _path(p):
    return spark._jvm.org.apache.hadoop.fs.Path(p)


def s3_exists(path):
    return _fs(path).exists(_path(path))


def s3_delete(path):
    fs = _fs(path)
    if fs.exists(_path(path)):
        fs.delete(_path(path), True)
        print(f"  deleted: {path}")
    else:
        print(f"  skip (not present): {path}")


def s3_list_files(path):
    """Recursive (relative_path, size) listing of files under path. [] if absent."""
    fs = _fs(path)
    if not fs.exists(_path(path)):
        return []
    base = _path(path).toString().rstrip("/")
    out = []
    it = fs.listFiles(_path(path), True)
    while it.hasNext():
        st = it.next()
        full = st.getPath().toString()
        out.append((full[len(base) + 1:] if full.startswith(base + "/") else full, st.getLen()))
    return sorted(out)


def s3_read_text(path):
    fs = _fs(path)
    if not fs.exists(_path(path)):
        raise FileNotFoundError(path)
    reader = spark._jvm.java.io.BufferedReader(
        spark._jvm.java.io.InputStreamReader(fs.open(_path(path)), "UTF-8")
    )
    try:
        lines, line = [], reader.readLine()
        while line is not None:
            lines.append(line)
            line = reader.readLine()
    finally:
        reader.close()
    return "\n".join(lines)


def s3_put_bytes(path, payload):
    """Write raw bytes to S3 via Hadoop FS (used for the Excel config)."""
    out = _fs(path).create(_path(path), True)
    try:
        out.write(bytearray(payload))
    finally:
        out.close()
    return len(payload)


# ── metastore accessors ──────────────────────────────────────────────────────────
def describe_formatted(qualified):
    """DESCRIBE FORMATTED as {col_name: data_type}, detail rows only."""
    out = {}
    for r in spark.sql(f"DESCRIBE FORMATTED {qualified}").collect():
        col = (r["col_name"] or "").strip()
        if col and not col.startswith("#"):
            out.setdefault(col, (r["data_type"] or "").strip())
    return out


def table_exists(db, tbl):
    try:
        return spark.sql(f"SHOW TABLES IN {db} LIKE '{tbl}'").count() > 0
    except Exception:
        return False


def location_of(qualified):
    return describe_formatted(qualified).get("Location")


def provider_of(qualified):
    d = describe_formatted(qualified)
    return (d.get("Provider") or d.get("Serde Library") or "").lower()


def table_type_of(qualified):
    d = describe_formatted(qualified)
    return (d.get("Type") or d.get("Table Type") or "").upper()


def count_of(qualified):
    return spark.sql(f"SELECT COUNT(*) AS c FROM {qualified}").collect()[0]["c"]


def columns_of(qualified):
    return {c["col_name"]: c["data_type"]
            for c in spark.sql(f"DESCRIBE {qualified}").collect()
            if c["col_name"] and not c["col_name"].startswith("#")}


# ── Airflow REST ─────────────────────────────────────────────────────────────────
_AIRFLOW_TOKEN = {"value": None}


def _airflow_token():
    if _AIRFLOW_TOKEN["value"]:
        return _AIRFLOW_TOKEN["value"]
    if not AIRFLOW_USERNAME or not AIRFLOW_PASSWORD:
        raise RuntimeError("Set AIRFLOW_USERNAME and AIRFLOW_PASSWORD before calling Airflow.")
    url = AIRFLOW_BASE_URL.rstrip("/") + "/auth/token"
    resp = requests.post(url, json={"username": AIRFLOW_USERNAME, "password": AIRFLOW_PASSWORD},
                         timeout=30)
    if not resp.ok:
        raise RuntimeError(f"POST {url} -> {resp.status_code}: {resp.text[:300]}")
    _AIRFLOW_TOKEN["value"] = resp.json()["access_token"]
    return _AIRFLOW_TOKEN["value"]


def airflow_request(method, path, **kwargs):
    headers = {"Authorization": f"Bearer {_airflow_token()}", **kwargs.pop("headers", {})}
    url = AIRFLOW_BASE_URL.rstrip("/") + path
    resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if resp.status_code == 401:
        _AIRFLOW_TOKEN["value"] = None
        headers["Authorization"] = f"Bearer {_airflow_token()}"
        resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if not resp.ok:
        raise RuntimeError(f"{method} {url} -> {resp.status_code}: {resp.text[:500]}")
    return resp.json() if resp.content else {}


print("Helpers ready.")

## Step 2 — Runtime assumption gate

Six assumptions the WF-374 design rests on. None can be checked from the repository — no Iceberg
version is pinned there, so what matters is what *this tenant's* Spark bundle does.

| # | Assumption | If it fails |
|---|---|---|
| 1 | `TBLPROPERTIES ('write.spark.fanout.enabled'='true')` is accepted and the CTAS succeeds from unsorted, many-partition input | Large partitioned text tables fail on write; fall back to a global `ORDER BY` on the partition columns |
| 2 | Renaming an **Iceberg** table is metastore-only — `Location` is unchanged | The swap would move data; the design does not hold |
| 3 | `DROP TABLE ... PURGE` works on an Iceberg table and removes its files | Failed-verification cleanup leaks a full copy every attempt |
| 4 | Renaming an **EXTERNAL Hive text** table leaves its `Location` alone | Parking the original as `_backup_` would trigger a full S3 copy |
| 5 | `DESCRIBE FORMATTED` reports a `Type` row | Every text table skips as `MANAGED_TEXT_INPLACE_UNSUPPORTED` |
| 6 | `CREATE TABLE ... LOCATION` over a path already holding Iceberg metadata **fails** | A retry could silently create a second table over the first copy's files |

This cell is self-contained: it creates and drops everything it needs under `PROBE_DB`, and can be
run on its own without the rest of the notebook.

In [ ]:
# ── Step 2 — Runtime assumption gate ────────────────────────────────────────────
PROBE_LOC = f"{TEST_BASE}/probe"
results = []


def probe(n, name, fn):
    try:
        detail = fn()
        results.append((n, True, name, detail))
        print(f"  PASS  {n}. {name}" + (f"  — {detail}" if detail else ""))
    except Exception as e:
        results.append((n, False, name, str(e)[:300]))
        print(f"  FAIL  {n}. {name}\n          {str(e)[:300]}")


spark.sql(f"DROP DATABASE IF EXISTS {PROBE_DB} CASCADE")
s3_delete(PROBE_LOC)
spark.sql(f"CREATE DATABASE {PROBE_DB} LOCATION '{PROBE_LOC}'")
spark.conf.set("hive.exec.dynamic.partition", "true")
spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")

# Unsorted, many-partition text source — the shape that needs fanout.
spark.sql(f"""
    CREATE EXTERNAL TABLE {PROBE_DB}.p_text (id BIGINT, name VARCHAR(20))
    PARTITIONED BY (dt STRING)
    ROW FORMAT DELIMITED FIELDS TERMINATED BY '|'
    STORED AS TEXTFILE
    LOCATION '{PROBE_LOC}/p_text'
""")
spark.sql(f"""
    INSERT INTO {PROBE_DB}.p_text PARTITION (dt)
    SELECT id, CONCAT('n', id) AS name,
           CONCAT('2026-01-', LPAD(CAST(id % 28 + 1 AS STRING), 2, '0')) AS dt
    FROM range(2000) ORDER BY RAND()
""")
SRC_ROWS = count_of(f"{PROBE_DB}.p_text")
print(f"Seeded {PROBE_DB}.p_text: {SRC_ROWS} rows across "
      f"{spark.sql(f'SHOW PARTITIONS {PROBE_DB}.p_text').count()} partitions\n")


def a1_fanout():
    spark.sql(f"""
        CREATE TABLE {PROBE_DB}.p_ice
        USING iceberg
        PARTITIONED BY (dt)
        LOCATION '{PROBE_LOC}/p_ice'
        TBLPROPERTIES ('write.spark.fanout.enabled'='true')
        AS SELECT * FROM {PROBE_DB}.p_text
    """)
    got = count_of(f"{PROBE_DB}.p_ice")
    assert got == SRC_ROWS, f"copied {got} of {SRC_ROWS} rows"
    props = describe_formatted(f"{PROBE_DB}.p_ice")
    kept = any("fanout" in str(v).lower() for v in props.values())
    return f"{got} rows copied; property visible in DESCRIBE: {kept}"


def a2_iceberg_rename_is_metadata_only():
    before = location_of(f"{PROBE_DB}.p_ice")
    spark.sql(f"ALTER TABLE {PROBE_DB}.p_ice RENAME TO {PROBE_DB}.p_ice_renamed")
    after = location_of(f"{PROBE_DB}.p_ice_renamed")
    assert before == after, f"location moved: {before} -> {after}"
    assert count_of(f"{PROBE_DB}.p_ice_renamed") == SRC_ROWS, "rows lost in rename"
    return f"location unchanged at {after}"


def a3_purge_removes_files():
    loc = location_of(f"{PROBE_DB}.p_ice_renamed")
    assert s3_list_files(loc), "no files to purge — probe is inconclusive"
    spark.sql(f"DROP TABLE {PROBE_DB}.p_ice_renamed PURGE")
    left = s3_list_files(loc)
    assert not left, f"{len(left)} file(s) survived PURGE at {loc}"
    return f"files removed from {loc}"


def a4_external_hive_rename_keeps_location():
    before = location_of(f"{PROBE_DB}.p_text")
    spark.sql(f"ALTER TABLE {PROBE_DB}.p_text RENAME TO {PROBE_DB}.p_text_backup_")
    after = location_of(f"{PROBE_DB}.p_text_backup_")
    assert before == after, f"metastore MOVED the data: {before} -> {after}"
    spark.sql(f"ALTER TABLE {PROBE_DB}.p_text_backup_ RENAME TO {PROBE_DB}.p_text")
    return f"location unchanged at {after}"


def a5_type_row_present():
    t = table_type_of(f"{PROBE_DB}.p_text")
    assert "EXTERNAL" in t, f"Type row reads {t or '(absent)'} — EXTERNAL detection would fail"
    return f"Type = {t}"


def a6_create_over_existing_metadata_fails():
    spark.sql(f"""
        CREATE TABLE {PROBE_DB}.p_occupied USING iceberg
        LOCATION '{PROBE_LOC}/occupied'
        AS SELECT 1 AS id
    """)
    # Drop the metastore entry only; the Iceberg metadata must stay on S3.
    spark.sql(f"DROP TABLE {PROBE_DB}.p_occupied")
    if not s3_list_files(f"{PROBE_LOC}/occupied"):
        return "inconclusive — plain DROP also removed the files, so the collision is unreachable"
    try:
        spark.sql(f"""
            CREATE TABLE {PROBE_DB}.p_occupied2 USING iceberg
            LOCATION '{PROBE_LOC}/occupied'
            AS SELECT 2 AS id
        """)
    except Exception as e:
        return f"correctly refused: {type(e).__name__}"
    raise AssertionError("CREATE over existing Iceberg metadata SUCCEEDED — "
                         "two tables may now share one location")


probe(1, "fanout property accepted on a partitioned CTAS", a1_fanout)
probe(2, "Iceberg RENAME is metastore-only", a2_iceberg_rename_is_metadata_only)
probe(3, "DROP ... PURGE removes an Iceberg table's files", a3_purge_removes_files)
probe(4, "EXTERNAL Hive rename keeps its location", a4_external_hive_rename_keeps_location)
probe(5, "DESCRIBE FORMATTED reports a Type row", a5_type_row_present)
probe(6, "CREATE over existing Iceberg metadata fails", a6_create_over_existing_metadata_fails)

failed = [r for r in results if not r[1]]
print()
if failed:
    raise RuntimeError(
        f"{len(failed)} runtime assumption(s) failed: {[r[0] for r in failed]}. "
        "Stop here — the WF-374 design depends on these and must be revisited before merge."
    )
print("All six runtime assumptions hold on this bundle. Safe to continue.")

## Step 3 — Bind tracking config from the *deployed* env file

`get_config()` resolves Airflow Variable → env var → default. The env files live beside the
deployed bundle, so reading them here means the notebook queries the same tracking database the
DAG writes to, instead of guessing.

In [ ]:
def env_from(path):
    out = {}
    try:
        text = s3_read_text(path)
    except FileNotFoundError:
        print(f"  (absent) {path}")
        return out
    for line in text.splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            out[k.strip()] = v.strip().strip('"').strip("'")
    print(f"  read {len(out)} keys from {path}")
    return out


_env = {}
_env.update(env_from(f"{CONFIG_S3_DIR}/env.shared"))
_env.update(env_from(f"{CONFIG_S3_DIR}/env.migration_dag_iceberg"))

TRACKING_DB     = _env.get("MIGRATION_TRACKING_DATABASE", "migration_tracking")
REPORT_LOCATION = _env.get("MIGRATION_REPORT_LOCATION", f"s3a://{BUCKET}/migration_reports")

print(f"\n  TRACKING_DB     = {TRACKING_DB}")
print(f"  REPORT_LOCATION = {REPORT_LOCATION}")
print("\nNote: an Airflow Variable of the same name outranks these. If the assertions cannot find "
      "tracking rows, check Variables first.")

## Step 4 — Privilege probe

Every later step needs create / insert / rename / drop in `TEST_DB`. A Ranger denial surfaces as a
`Py4JJavaError` wrapping an `AccessControlException`, which is easy to mistake for a bug in the
DAG. Fail here instead, with the missing grant named.

In [ ]:
from py4j.protocol import Py4JJavaError

PROBE_TBL = "_wf374_authz_probe"
denials = []


def try_sql(label, sql):
    try:
        spark.sql(sql)
        print(f"  ok    {label}")
    except Py4JJavaError as e:
        msg = str(e.java_exception)[:200] if e.java_exception else str(e)[:200]
        denials.append((label, msg))
        print(f"  DENIED {label}\n           {msg}")
    except Exception as e:
        denials.append((label, str(e)[:200]))
        print(f"  ERROR  {label}\n           {str(e)[:200]}")


try_sql("CREATE DATABASE", f"CREATE DATABASE IF NOT EXISTS {TEST_DB} LOCATION '{TEST_BASE}/{TEST_DB}'")
try_sql("CREATE TABLE",    f"CREATE EXTERNAL TABLE IF NOT EXISTS {TEST_DB}.{PROBE_TBL} (id INT) "
                           f"STORED AS TEXTFILE LOCATION '{TEST_BASE}/{PROBE_TBL}'")
try_sql("INSERT",          f"INSERT INTO {TEST_DB}.{PROBE_TBL} VALUES (1)")
try_sql("RENAME",          f"ALTER TABLE {TEST_DB}.{PROBE_TBL} RENAME TO {TEST_DB}.{PROBE_TBL}_r")
try_sql("RENAME BACK",     f"ALTER TABLE {TEST_DB}.{PROBE_TBL}_r RENAME TO {TEST_DB}.{PROBE_TBL}")
try_sql("DROP TABLE",      f"DROP TABLE IF EXISTS {TEST_DB}.{PROBE_TBL} PURGE")

if denials:
    raise RuntimeError(f"{len(denials)} privilege(s) missing: {[d[0] for d in denials]}. "
                       "Ask for the grants, or point TEST_DB at a namespace you own.")
print("\nAll required privileges present.")

## Step 5 — Seed the matrix

Seven tables, one per routing case. Baselines (row counts, locations, schemas) are captured now
so the assertions can prove the copy is faithful rather than merely present.

| table | shape | expected outcome |
|---|---|---|
| `t_plain` | EXTERNAL text, unpartitioned | `INPLACE_CTAS` |
| `t_part` | EXTERNAL text, partitioned by `dt` | `INPLACE_CTAS`, partition counts match |
| `t_varchar` | EXTERNAL text with `varchar(20)` / `char(3)` | `INPLACE_CTAS` — the case that fails without char/varchar normalization |
| `t_empty` | EXTERNAL text, zero rows | `INPLACE_CTAS`, trivially valid |
| `t_managed` | MANAGED text | SKIPPED `MANAGED_TEXT_INPLACE_UNSUPPORTED` |
| `t_parquet` | EXTERNAL parquet | `INPLACE` via `system.migrate`, zero-copy |
| `t_snap` (in `SNAP_DB`) | EXTERNAL text, `inplace_migration=F` | snapshot CTAS into `{SNAP_DB}_iceberg` |

In [ ]:
# ── Step 5 — Seed the matrix ────────────────────────────────────────────────────
for db in (TEST_DB, SNAP_DB, f"{SNAP_DB}_iceberg"):
    spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
s3_delete(f"{TEST_BASE}/{TEST_DB}")
s3_delete(f"{TEST_BASE}/{SNAP_DB}")

spark.sql(f"CREATE DATABASE {TEST_DB} LOCATION '{TEST_BASE}/{TEST_DB}'")
spark.sql(f"CREATE DATABASE {SNAP_DB} LOCATION '{TEST_BASE}/{SNAP_DB}'")


def text_table(db, tbl, cols, partitioned_by=None, location=None, managed=False):
    loc = location or f"{TEST_BASE}/{db}/{tbl}"
    part = f"PARTITIONED BY ({partitioned_by})" if partitioned_by else ""
    if managed:
        spark.sql(f"CREATE TABLE {db}.{tbl} ({cols}) {part} "
                  f"ROW FORMAT DELIMITED FIELDS TERMINATED BY '|' STORED AS TEXTFILE")
    else:
        spark.sql(f"CREATE EXTERNAL TABLE {db}.{tbl} ({cols}) {part} "
                  f"ROW FORMAT DELIMITED FIELDS TERMINATED BY '|' STORED AS TEXTFILE "
                  f"LOCATION '{loc}'")


text_table(TEST_DB, "t_plain", "id BIGINT, amount DOUBLE, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_plain SELECT id, id * 1.5, CONCAT('n', id) FROM range(500)")

text_table(TEST_DB, "t_part", "id BIGINT, amount DOUBLE", partitioned_by="dt STRING, region STRING")
for dt, region in (("2026-01-01", "eu"), ("2026-01-01", "us"), ("2026-01-02", "eu")):
    spark.sql(f"INSERT INTO {TEST_DB}.t_part PARTITION (dt='{dt}', region='{region}') "
              f"SELECT id, id * 2.0 FROM range(300)")

text_table(TEST_DB, "t_varchar", "id BIGINT, name VARCHAR(20), code CHAR(3)")
spark.sql(f"INSERT INTO {TEST_DB}.t_varchar SELECT id, CONCAT('name', id), 'abc' FROM range(200)")

text_table(TEST_DB, "t_empty", "id BIGINT, note STRING")

text_table(TEST_DB, "t_managed", "id BIGINT, note STRING", managed=True)
spark.sql(f"INSERT INTO {TEST_DB}.t_managed SELECT id, CONCAT('m', id) FROM range(50)")

spark.sql(f"CREATE EXTERNAL TABLE {TEST_DB}.t_parquet (id BIGINT, amount DOUBLE) "
          f"STORED AS PARQUET LOCATION '{TEST_BASE}/{TEST_DB}/t_parquet'")
spark.sql(f"INSERT INTO {TEST_DB}.t_parquet SELECT id, id * 3.0 FROM range(400)")

text_table(SNAP_DB, "t_snap", "id BIGINT, note STRING",
           location=f"{TEST_BASE}/{SNAP_DB}/t_snap")
spark.sql(f"INSERT INTO {SNAP_DB}.t_snap SELECT id, CONCAT('s', id) FROM range(100)")

INPLACE_TABLES = ["t_plain", "t_part", "t_varchar", "t_empty", "t_managed", "t_parquet"]
BASELINE = {}
for t in INPLACE_TABLES:
    q = f"{TEST_DB}.{t}"
    BASELINE[t] = {"rows": count_of(q), "location": location_of(q),
                   "columns": columns_of(q), "type": table_type_of(q)}
BASELINE["t_snap"] = {"rows": count_of(f"{SNAP_DB}.t_snap"),
                      "location": location_of(f"{SNAP_DB}.t_snap"),
                      "columns": columns_of(f"{SNAP_DB}.t_snap"),
                      "type": table_type_of(f"{SNAP_DB}.t_snap")}

print("\nSeeded:")
for t, b in BASELINE.items():
    print(f"  {t:<12} rows={b['rows']:<6} type={b['type']:<16} {b['location']}")

## Step 6 — Excel config and run helpers

Columns are `database | table | inplace_migration | destination_iceberg_database`. Blank
destination means the DAG derives it: the source database for in-place, `{db}_iceberg` for
snapshot.

`run_matrix` writes an Excel, triggers a run, waits, and returns the tracking rows — the later
scenarios each call it with one or two rows.

In [ ]:
from io import BytesIO
from datetime import datetime, timezone
import time

import pandas as pd

TERMINAL = {"success", "failed"}
RUNS = {}


def write_excel(rows, tag):
    """rows: list of (database, table, inplace_migration, destination_iceberg_database)."""
    df = pd.DataFrame(rows, columns=["database", "table", "inplace_migration",
                                     "destination_iceberg_database"])
    buf = BytesIO()
    df.to_excel(buf, index=False, engine="openpyxl")
    path = f"{EXCEL_DIR}/wf374_{_tag}_{tag}.xlsx"
    s3_put_bytes(path, buf.getvalue())
    print(f"  wrote {path}  ({len(rows)} row(s))")
    return path


def trigger_run(tag, conf):
    logical_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
    run_id = f"manual_wf374_{tag}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
    resp = airflow_request("POST", f"/api/v2/dags/{DAG_ID}/dagRuns", json={
        "dag_run_id": run_id, "logical_date": logical_date, "conf": conf,
    })
    print(f"  triggered {resp['dag_run_id']}  conf={conf}")
    return resp["dag_run_id"]


def wait_for_run(dag_run_id, timeout=60 * 30):
    deadline, last = time.time() + timeout, None
    while time.time() < deadline:
        state = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{dag_run_id}").get("state")
        if state != last:
            print(f"  [{time.strftime('%H:%M:%S')}] state = {state}")
            last = state
        if state in TERMINAL:
            return state
        time.sleep(15)
    print(f"  timed out after {timeout}s")
    return last


def load_tracking(dag_run_id):
    """Return (run_row_dict, per-table pandas df, internal run_id)."""
    runs = spark.sql(
        f"SELECT * FROM {TRACKING_DB}.iceberg_migration_runs "
        f"WHERE dag_run_id = '{dag_run_id}' ORDER BY started_at DESC"
    ).collect()
    assert runs, f"No iceberg_migration_runs row for dag_run_id={dag_run_id}"
    rid = runs[0]["run_id"]
    status_pd = spark.sql(
        "SELECT source_database, source_table, migration_type, destination_database, "
        "destination_table, table_location, status, source_hive_row_count, "
        "destination_iceberg_row_count, row_count_match, source_hive_partition_count, "
        "dest_iceberg_partition_count, partition_count_match, schema_match, "
        "validation_status, error_message "
        f"FROM {TRACKING_DB}.iceberg_migration_table_status "
        f"WHERE run_id = '{rid}' ORDER BY source_table"
    ).toPandas()
    return runs[0].asDict(), status_pd, rid


def run_matrix(rows, tag, conf_extra=None):
    excel = write_excel(rows, tag)
    conf = {"excel_file_path": excel}
    conf.update(conf_extra or {})
    dag_run_id = trigger_run(tag, conf)
    state = wait_for_run(dag_run_id)
    run_row, status_pd, rid = load_tracking(dag_run_id)
    RUNS[tag] = {"dag_run_id": dag_run_id, "run_id": rid, "state": state, "excel": excel}
    print(f"  run state = {state}, internal run_id = {rid}")
    return status_pd


def rows_by_table(status_pd):
    return {r["source_table"]: r for _, r in status_pd.iterrows()}


def reason_of(row):
    """Reason code from an error_message formatted '[CODE] detail'."""
    msg = row.get("error_message") or ""
    return msg[1:msg.index("]")] if msg.startswith("[") and "]" in msg else ""


failures = []


def check(cond, msg):
    if cond:
        print(f"  PASS  {msg}")
    else:
        failures.append(msg)
        print(f"  FAIL  {msg}")


print("Run helpers ready.")

## Step 7 — Run 1: the full matrix, flag at its default (on)

One run covers every routing row. The in-place group and the snapshot group are separate Excel
rows, so `parse_iceberg_excel` groups them into two mapped task instances.

In [ ]:
BASE_ROWS = [
    (TEST_DB, "t_plain",   "T", ""),
    (TEST_DB, "t_part",    "T", ""),
    (TEST_DB, "t_varchar", "T", ""),
    (TEST_DB, "t_empty",   "T", ""),
    (TEST_DB, "t_managed", "T", ""),
    (TEST_DB, "t_parquet", "T", ""),
    (SNAP_DB, "t_snap",    "F", ""),
]

status_1 = run_matrix(BASE_ROWS, "happy")
status_1

## Step 8 — Assertions on run 1

The routing matrix, expressed as assertions. `t_managed` and `t_parquet` are the two guards that
prove the change did not widen its blast radius: a table that cannot be confirmed EXTERNAL must be
skipped, and a Parquet table must still take the zero-copy path.

In [ ]:
r1 = rows_by_table(status_1)
print("Run 1 — routing matrix\n")

for t in ("t_plain", "t_part", "t_varchar", "t_empty"):
    row = r1.get(t)
    check(row is not None, f"{t}: has a tracking row")
    if row is None:
        continue
    check(row["status"] == "COMPLETED", f"{t}: status COMPLETED (got {row['status']})")
    check(row["migration_type"] == "INPLACE_CTAS",
          f"{t}: migration_type INPLACE_CTAS (got {row['migration_type']})")
    check(row["destination_database"] == TEST_DB,
          f"{t}: destination database is the source database")
    check(bool(row["row_count_match"]), f"{t}: row counts match")
    check(str(row["table_location"] or "").endswith("_iceberg"),
          f"{t}: landed at a _iceberg location (got {row['table_location']})")

managed = r1.get("t_managed")
check(managed is not None and managed["status"] == "SKIPPED", "t_managed: SKIPPED")
check(managed is not None and reason_of(managed) == "MANAGED_TEXT_INPLACE_UNSUPPORTED",
      f"t_managed: reason MANAGED_TEXT_INPLACE_UNSUPPORTED (got {reason_of(managed) if managed is not None else 'no row'})")

parq = r1.get("t_parquet")
check(parq is not None and parq["status"] == "COMPLETED", "t_parquet: COMPLETED")
check(parq is not None and parq["migration_type"] == "INPLACE",
      f"t_parquet: still zero-copy INPLACE (got {parq['migration_type'] if parq is not None else 'no row'})")

snap = r1.get("t_snap")
check(snap is not None and snap["status"] == "COMPLETED", "t_snap: COMPLETED")
check(snap is not None and snap["migration_type"] == "SNAPSHOT",
      f"t_snap: snapshot mode unchanged (got {snap['migration_type'] if snap is not None else 'no row'})")
check(snap is not None and snap["destination_database"] == f"{SNAP_DB}_iceberg",
      "t_snap: destination is the _iceberg database")

part = r1.get("t_part")
check(part is not None and bool(part["partition_count_match"]), "t_part: partition counts match")

print(f"\n{len(failures)} failure(s) so far.")

## Step 9 — Verify the swap directly

Tracking records the DAG's own conclusions. This queries the metastore and S3 to confirm them
independently: the table is Iceberg under its original name, the backup still holds the original
text data, and the copy is faithful.

In [ ]:
print("Run 1 — direct verification\n")

for t in ("t_plain", "t_part", "t_varchar", "t_empty"):
    q, b = f"{TEST_DB}.{t}", BASELINE[t]
    check("iceberg" in provider_of(q), f"{t}: provider is iceberg")
    check(count_of(q) == b["rows"], f"{t}: {count_of(q)} rows == baseline {b['rows']}")
    check(str(location_of(q) or "") == f"{b['location'].rstrip('/')}_iceberg",
          f"{t}: location is <original>_iceberg")

    backup = f"{TEST_DB}.{t}_backup_"
    check(table_exists(TEST_DB, f"{t}_backup_"), f"{t}: backup table retained")
    if table_exists(TEST_DB, f"{t}_backup_"):
        check(location_of(backup) == b["location"], f"{t}: backup still points at the original path")
        check(count_of(backup) == b["rows"], f"{t}: backup still reads {b['rows']} original rows")
        check(bool(s3_list_files(b["location"])), f"{t}: original text files still on S3")

    check(not table_exists(TEST_DB, f"{t}__ice_staging"), f"{t}: no staging table left behind")
    check(set(columns_of(q)) == set(b["columns"]), f"{t}: column set unchanged")

# The char/varchar case: Iceberg has no varchar, so these must read back as string —
# and must NOT have been treated as a schema mismatch.
vc = columns_of(f"{TEST_DB}.t_varchar")
check(vc.get("name") == "string", f"t_varchar: varchar(20) -> string (got {vc.get('name')})")
check(vc.get("code") == "string", f"t_varchar: char(3) -> string (got {vc.get('code')})")
check(count_of(f"{TEST_DB}.t_varchar WHERE name IS NOT NULL") == BASELINE["t_varchar"]["rows"],
      "t_varchar: no values lost to NULL in the copy")

# t_managed must be untouched: still text, still managed, no backup, no staging.
check("lazysimpleserde" in provider_of(f"{TEST_DB}.t_managed"), "t_managed: still a text table")
check(not table_exists(TEST_DB, "t_managed_backup_"), "t_managed: no backup created")
check(not table_exists(TEST_DB, "t_managed__ice_staging"), "t_managed: no staging created")

print(f"\n{len(failures)} failure(s) so far.")

## Step 10 — Re-run: idempotence

A second run over the same Excel must recognise the tables as already Iceberg and do nothing. This
also covers the retry-safety fix: `_repair_partial_text_swap` sees a live Iceberg table beside its
`_backup_` and returns `ALREADY_MIGRATED` instead of mistaking it for a name collision.

In [ ]:
status_2 = run_matrix(BASE_ROWS, "rerun")
r2 = rows_by_table(status_2)
print("\nRun 2 — idempotence\n")

for t in ("t_plain", "t_part", "t_varchar", "t_empty"):
    row = r2.get(t)
    check(row is not None and row["status"] == "SKIPPED", f"{t}: SKIPPED on re-run")
    check(row is not None and reason_of(row) == "ALREADY_ICEBERG",
          f"{t}: reason ALREADY_ICEBERG (got {reason_of(row) if row is not None else 'no row'})")
    check(count_of(f"{TEST_DB}.{t}") == BASELINE[t]["rows"], f"{t}: row count still correct")
    check(not table_exists(TEST_DB, f"{t}__ice_staging"), f"{t}: no staging table created")

print(f"\n{len(failures)} failure(s) so far.")

## Step 11 — Flag off

`iceberg_inplace_text_ctas = false` in the run conf must restore the old behaviour: the table is
skipped with `TEXT_FORMAT_INPLACE_UNSUPPORTED`, nothing is copied, and the message names both
routes rather than presenting the namespace change as the only option.

In [ ]:
text_table(TEST_DB, "t_flagoff", "id BIGINT, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_flagoff SELECT id, CONCAT('f', id) FROM range(30)")
BASELINE["t_flagoff"] = {"rows": count_of(f"{TEST_DB}.t_flagoff"),
                         "location": location_of(f"{TEST_DB}.t_flagoff")}

status_3 = run_matrix([(TEST_DB, "t_flagoff", "T", "")], "flagoff",
                      conf_extra={"iceberg_inplace_text_ctas": False})
r3 = rows_by_table(status_3)
print("\nRun 3 — flag off\n")

row = r3.get("t_flagoff")
check(row is not None and row["status"] == "SKIPPED", "t_flagoff: SKIPPED")
check(row is not None and reason_of(row) == "TEXT_FORMAT_INPLACE_UNSUPPORTED",
      f"t_flagoff: reason TEXT_FORMAT_INPLACE_UNSUPPORTED (got {reason_of(row) if row is not None else 'no row'})")
check("lazysimpleserde" in provider_of(f"{TEST_DB}.t_flagoff"), "t_flagoff: still a text table")
check(not table_exists(TEST_DB, "t_flagoff__ice_staging"), "t_flagoff: no staging table created")
check(not table_exists(TEST_DB, "t_flagoff_backup_"), "t_flagoff: no backup created")
if row is not None:
    print(f"\n  message: {row['error_message']}")

print(f"\n{len(failures)} failure(s) so far.")

## Step 12 — Verification gate (replay)

The gate cannot be forced deterministically through the DAG: it fires when the source changes
between the pre-copy row count and the CTAS, which is a race. This cell **replays the gate's exact
statements by hand** against a table whose copy is deliberately short, and proves the two
properties that matter — the staging table is purged, and the original is untouched.

The gate logic itself is covered by unit tests
(`TestMigrateTextTableInplace::test_row_mismatch_purges_staging_and_does_not_rename`). The optional
cell after this one attempts the real race, for anyone who wants it.

In [ ]:
text_table(TEST_DB, "t_gate", "id BIGINT, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_gate SELECT id, CONCAT('g', id) FROM range(100)")
gate_loc = location_of(f"{TEST_DB}.t_gate")
gate_rows = count_of(f"{TEST_DB}.t_gate")
staging = f"{TEST_DB}.t_gate__ice_staging"

print("Replaying the gate with a deliberately short copy\n")

# 1. the copy — one row short of the source, standing in for a concurrent delete
spark.sql(f"""
    CREATE TABLE {staging} USING iceberg
    LOCATION '{gate_loc}_iceberg'
    TBLPROPERTIES ('write.spark.fanout.enabled'='true')
    AS SELECT * FROM {TEST_DB}.t_gate WHERE id > 0
""")
staging_rows = count_of(staging)
print(f"  source rows  = {gate_rows}")
print(f"  staging rows = {staging_rows}")

# 2. the gate's own comparison
gate_fails = staging_rows != gate_rows
check(gate_fails, "gate detects the mismatch (staging_rows != hive_count)")

# 3. the gate's failure path
if gate_fails:
    spark.sql(f"DROP TABLE IF EXISTS {staging} PURGE")

check(not table_exists(TEST_DB, "t_gate__ice_staging"), "staging table dropped")
check(not s3_list_files(f"{gate_loc}_iceberg"), "staging files purged from S3")
check("lazysimpleserde" in provider_of(f"{TEST_DB}.t_gate"), "original still a text table")
check(count_of(f"{TEST_DB}.t_gate") == gate_rows, "original still has all its rows")
check(not table_exists(TEST_DB, "t_gate_backup_"), "original was never renamed")

print(f"\n{len(failures)} failure(s) so far.")

### Optional — the real race

Runs the DAG against a table large enough for the copy to take measurable time, and inserts rows
while it runs. Whether the gate fires depends on timing, so treat a pass as confirmation and a
non-fire as inconclusive, never as a failure. Skip this cell unless you want the real thing.

In [ ]:
RUN_RACE = False   # flip to True to attempt it

if RUN_RACE:
    import threading

    text_table(TEST_DB, "t_race", "id BIGINT, note STRING")
    spark.sql(f"INSERT INTO {TEST_DB}.t_race SELECT id, CONCAT('r', id) FROM range(3000000)")
    print(f"  seeded {count_of(f'{TEST_DB}.t_race')} rows")

    excel = write_excel([(TEST_DB, "t_race", "T", "")], "race")
    dag_run_id = trigger_run("race", {"excel_file_path": excel})

    def writer():
        time.sleep(45)          # aim for the middle of the CTAS
        spark.sql(f"INSERT INTO {TEST_DB}.t_race SELECT id, 'late' FROM range(1000)")
        print("  injected 1000 rows mid-run")

    threading.Thread(target=writer, daemon=True).start()
    wait_for_run(dag_run_id)
    _, race_pd, _ = load_tracking(dag_run_id)
    row = rows_by_table(race_pd).get("t_race")
    code_ = reason_of(row) if row is not None else ""
    if code_ == "INPLACE_CTAS_VERIFY_FAILED":
        print("  gate fired on a real concurrent write — original left untouched:",
              "lazysimpleserde" in provider_of(f"{TEST_DB}.t_race"))
    else:
        print(f"  inconclusive — the write missed the window (status={row['status'] if row is not None else 'n/a'}, "
              f"reason={code_ or 'none'}). Not a failure.")
else:
    print("Skipped (RUN_RACE = False).")

## Step 13 — Repair states

The states `_repair_partial_text_swap` can find, each constructed by hand and then handed to a
real DAG run. These paths only execute when something has already gone wrong, which is exactly why
they are worth testing.

| state constructed | expected |
|---|---|
| live text table + leftover `__ice_staging` | staging purged, then migrated normally |
| table parked as `_backup_`, base name missing | rolled back, then migrated normally |
| live text table + unrelated `_backup_` elsewhere | `INPLACE_CTAS_BACKUP_CONFLICT` (`SKIPPED`), nothing touched |
| base name missing, no backup | `INPLACE_CTAS_SWAP_INCOMPLETE` (`FAILED`), reported not silent |
| fresh re-run of a migrated table | `ALREADY_ICEBERG` — covered in Step 10 |

Two states are not reachable from here, both because they need a task retry inside a single run
while discovery still describes the table as live text:

- **base name missing, backup at an unexpected location** — recorded `FAILED` /
  `INPLACE_CTAS_SWAP_INCOMPLETE` rather than skipped, so a vanished table cannot finish green.
- **already swapped by an earlier attempt of the same run** — re-verified and left `COMPLETED`,
  rather than rewritten as a skip that would drop it out of validation.

On a fresh run discovery catches the first of these before the migration task sees the table, which
is what 13b exercises.


In [ ]:
print("=== 13a — leftover staging table is cleared, then the table migrates ===\n")
text_table(TEST_DB, "t_stale", "id BIGINT, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_stale SELECT id, CONCAT('s', id) FROM range(40)")
stale_loc, stale_rows = location_of(f"{TEST_DB}.t_stale"), count_of(f"{TEST_DB}.t_stale")

# a staging table an interrupted attempt would have left behind
spark.sql(f"CREATE TABLE {TEST_DB}.t_stale__ice_staging USING iceberg "
          f"LOCATION '{stale_loc}_iceberg' AS SELECT * FROM {TEST_DB}.t_stale WHERE id < 5")
print(f"  planted staging with {count_of(f'{TEST_DB}.t_stale__ice_staging')} rows")

row = rows_by_table(run_matrix([(TEST_DB, "t_stale", "T", "")], "stale")).get("t_stale")
check(row is not None and row["status"] == "COMPLETED", "t_stale: migrated despite the leftover")
check(row is not None and row["migration_type"] == "INPLACE_CTAS", "t_stale: INPLACE_CTAS")
check(count_of(f"{TEST_DB}.t_stale") == stale_rows,
      f"t_stale: {stale_rows} rows — the partial staging copy was discarded, not promoted")
check(not table_exists(TEST_DB, "t_stale__ice_staging"), "t_stale: no staging table left")
print(f"\n{len(failures)} failure(s) so far.")

In [ ]:
print("=== 13b — a swap interrupted between the renames is rolled back ===\n")
text_table(TEST_DB, "t_rollback", "id BIGINT, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_rollback SELECT id, CONCAT('b', id) FROM range(60)")
rb_rows = count_of(f"{TEST_DB}.t_rollback")

# exactly the state left by a crash between the two renames
spark.sql(f"ALTER TABLE {TEST_DB}.t_rollback RENAME TO {TEST_DB}.t_rollback_backup_")
print(f"  parked as t_rollback_backup_; t_rollback exists = {table_exists(TEST_DB, 't_rollback')}")

# Discovery cannot see a table that is not in the metastore, so on a FRESH run it reports the
# interrupted swap rather than migrating it. That report is the deliverable here.
row = rows_by_table(run_matrix([(TEST_DB, "t_rollback", "T", "")], "rollback")).get("t_rollback")
check(row is not None, "t_rollback: reported rather than silently vanishing")
check(row is not None and reason_of(row) == "INPLACE_CTAS_SWAP_INCOMPLETE",
      f"t_rollback: reason INPLACE_CTAS_SWAP_INCOMPLETE (got {reason_of(row) if row is not None else 'no row'})")
check(row is not None and row["status"] == "FAILED", "t_rollback: FAILED, so it needs attention")
if row is not None:
    print(f"\n  operator message:\n  {row['error_message']}\n")

# follow the recovery the message prints, then confirm the table migrates cleanly
spark.sql(f"ALTER TABLE {TEST_DB}.t_rollback_backup_ RENAME TO {TEST_DB}.t_rollback")
spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.t_rollback__ice_staging PURGE")
row = rows_by_table(run_matrix([(TEST_DB, "t_rollback", "T", "")], "rollback2")).get("t_rollback")
check(row is not None and row["status"] == "COMPLETED", "t_rollback: migrates after the documented recovery")
check(count_of(f"{TEST_DB}.t_rollback") == rb_rows, f"t_rollback: {rb_rows} rows intact")
print(f"\n{len(failures)} failure(s) so far.")

In [ ]:
print("=== 13c — an unrelated _backup_ table is never touched ===\n")
text_table(TEST_DB, "t_conflict", "id BIGINT, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_conflict SELECT id, CONCAT('c', id) FROM range(20)")

# a table the tenant happens to own, at a location that has nothing to do with t_conflict
text_table(TEST_DB, "t_conflict_backup_", "id BIGINT, other STRING",
           location=f"{TEST_BASE}/{TEST_DB}/unrelated_backup")
spark.sql(f"INSERT INTO {TEST_DB}.t_conflict_backup_ SELECT id, 'precious' FROM range(7)")
precious_rows = count_of(f"{TEST_DB}.t_conflict_backup_")
precious_loc = location_of(f"{TEST_DB}.t_conflict_backup_")

row = rows_by_table(run_matrix([(TEST_DB, "t_conflict", "T", "")], "conflict")).get("t_conflict")
check(row is not None and row["status"] == "SKIPPED", "t_conflict: SKIPPED, not migrated over")
check(row is not None and reason_of(row) == "INPLACE_CTAS_BACKUP_CONFLICT",
      f"t_conflict: reason INPLACE_CTAS_BACKUP_CONFLICT (got {reason_of(row) if row is not None else 'no row'})")
check(count_of(f"{TEST_DB}.t_conflict_backup_") == precious_rows,
      "the tenant's own _backup_ table still has all its rows")
check(location_of(f"{TEST_DB}.t_conflict_backup_") == precious_loc,
      "the tenant's own _backup_ table was not renamed or moved")
check("lazysimpleserde" in provider_of(f"{TEST_DB}.t_conflict"), "t_conflict: left as text")
check(not table_exists(TEST_DB, "t_conflict__ice_staging"), "t_conflict: no staging created")
print(f"\n{len(failures)} failure(s) so far.")

## Step 14 — `iceberg_drop_backup = true`

The backup is dropped from the metastore, but its text files must survive on S3: they are tenant
data, not this run's output, and nothing in the DAG is allowed to delete them. The orphaned files
are a documented manual-cleanup cost, not a bug.

In [ ]:
text_table(TEST_DB, "t_dropbk", "id BIGINT, note STRING")
spark.sql(f"INSERT INTO {TEST_DB}.t_dropbk SELECT id, CONCAT('d', id) FROM range(80)")
dropbk_loc = location_of(f"{TEST_DB}.t_dropbk")
dropbk_rows = count_of(f"{TEST_DB}.t_dropbk")
files_before = len(s3_list_files(dropbk_loc))

row = rows_by_table(run_matrix([(TEST_DB, "t_dropbk", "T", "")], "dropbk",
                               conf_extra={"iceberg_drop_backup": True})).get("t_dropbk")
print()
check(row is not None and row["status"] == "COMPLETED", "t_dropbk: COMPLETED")
check("iceberg" in provider_of(f"{TEST_DB}.t_dropbk"), "t_dropbk: now Iceberg")
check(count_of(f"{TEST_DB}.t_dropbk") == dropbk_rows, f"t_dropbk: {dropbk_rows} rows")
check(not table_exists(TEST_DB, "t_dropbk_backup_"), "backup dropped from the metastore")
check(len(s3_list_files(dropbk_loc)) == files_before,
      f"original text files survive on S3 ({files_before} file(s)) — metadata-only drop, never PURGE")
print(f"\n  orphaned text files remain at: {dropbk_loc}")
print(f"\n{len(failures)} failure(s) so far.")

## Step 15 — Discovery excludes backup and staging tables

A `*` pattern must never pick up a `_backup_` or `__ice_staging` table as a migration candidate,
or a re-run would try to migrate the previous run's leftovers.

In [ ]:
# By now TEST_DB holds several _backup_ tables and no staging tables. Plant one so both
# exclusions are exercised, then run the whole database with a wildcard.
spark.sql(f"CREATE TABLE IF NOT EXISTS {TEST_DB}.t_plain__ice_staging USING iceberg "
          f"LOCATION '{TEST_BASE}/{TEST_DB}/orphan_staging' AS SELECT 1 AS id")

before = {r["tableName"] for r in spark.sql(f"SHOW TABLES IN {TEST_DB}").collect()}
backups = sorted(t for t in before if t.endswith("_backup_"))
stagings = sorted(t for t in before if t.endswith("__ice_staging"))
print(f"  {len(backups)} backup table(s), {len(stagings)} staging table(s) present\n")

status_w = run_matrix([(TEST_DB, "*", "T", "")], "wildcard")
seen = set(status_w["source_table"])
check(not (seen & set(backups)), f"no _backup_ table was picked up (backups: {backups})")
check(not (seen & set(stagings)), f"no __ice_staging table was picked up (staging: {stagings})")
check("t_plain" in seen, "the real tables were still discovered")

spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.t_plain__ice_staging PURGE")
print(f"\n{len(failures)} failure(s) so far.")

## Step 16 — HTML report

`generate_iceberg_html_report` writes `{report_location}/{run_id}_iceberg_report.html`. Confirm the
new reason codes and `INPLACE_CTAS` render for a human reader.

In [ ]:
from IPython.display import HTML, display

report_path = f"{REPORT_LOCATION}/{RUNS['happy']['run_id']}_iceberg_report.html"
try:
    html = s3_read_text(report_path)
    print(f"Report: {report_path}  ({len(html)} bytes)\n")
    for token in ("INPLACE_CTAS", "MANAGED_TEXT_INPLACE_UNSUPPORTED", "t_plain"):
        check(token in html, f"report mentions {token}")
    display(HTML(html))
except FileNotFoundError:
    print(f"No report at {report_path} — check REPORT_LOCATION against the deployed env file.")

## Step 17 — Verdict

In [ ]:
print("=" * 78)
if failures:
    print(f"{len(failures)} CHECK(S) FAILED\n")
    for f in failures:
        print(f"  - {f}")
else:
    print("ALL CHECKS PASSED")
print("=" * 78)
print("\nRuns triggered:")
for tag, r in RUNS.items():
    print(f"  {tag:<10} {r['state']:<8} dag_run_id={r['dag_run_id']}  run_id={r['run_id']}")

## Step 18 — Cleanup

Drops the test databases, wipes the seeded S3 tree, and removes the Excel configs, reports and
tracking rows this notebook created. Run it even after a failure — the artifacts are all named
with the suffix, so nothing else is at risk.

In [ ]:
print("Cleaning up test artifacts...\n")

for db in (TEST_DB, SNAP_DB, f"{SNAP_DB}_iceberg", PROBE_DB):
    try:
        for r in spark.sql(f"SHOW TABLES IN {db}").collect():
            spark.sql(f"DROP TABLE IF EXISTS {db}.{r['tableName']} PURGE")
        spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
        print(f"  dropped database: {db}")
    except Exception as e:
        print(f"  database cleanup skipped for {db}: {e}")

s3_delete(TEST_BASE)

for tag, r in RUNS.items():
    try:
        s3_delete(r["excel"])
    except Exception as e:
        print(f"  excel delete skipped for {tag}: {e}")
    try:
        s3_delete(f"{REPORT_LOCATION}/{r['run_id']}_iceberg_report.html")
    except Exception as e:
        print(f"  report delete skipped for {tag}: {e}")
    try:
        spark.sql(f"DELETE FROM {TRACKING_DB}.iceberg_migration_table_status "
                  f"WHERE run_id = '{r['run_id']}'")
        spark.sql(f"DELETE FROM {TRACKING_DB}.iceberg_migration_runs "
                  f"WHERE run_id = '{r['run_id']}'")
        print(f"  deleted tracking rows for {tag} ({r['run_id']})")
    except Exception as e:
        print(f"  tracking cleanup skipped for {tag}: {e}")

print("\nCleanup complete.")